# Использование пула потоков

ThreadPoolExecutor — это пул потоков, то есть заранее созданный набор рабочих потоков, которым можно поручать выполнение задач.

Когда нужно выполнить синхронную или блокирующую функцию, вызывается `loop.run_in_executor(executor, func, *args)`:

1. Асинхронная корутина вызывает run_in_executor(), которая говорит event-loop "Выполни эту синхронную функцию в отдельном потоке"
2. Задача отправляется в пул потоков (ThreadPoolExecutor): Пул берёт свободный поток и запускает в нём обычную синхронную функцию.
3. Пока поток занят, event-loop НЕ блокируется: Луп продолжает обслуживать другие корутины, таймеры, сокеты и т.п.
4. После завершения работы потока — результат возвращается: Поток заполняет объект Future, и asyncio возобновляет корутину с готовым значением

`asyncio` работает в одном потоке, `ThreadPoolExecutor` создаёт дополнительные потоки, а ОС распределяет CPU-время между ними

## Пример 1

In [4]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
import sqlite3

In [5]:
# Создаем базу
conn = sqlite3.connect("demo.db")
cursor = conn.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS items (id INTEGER PRIMARY KEY, value TEXT)")
conn.commit()
conn.close()

In [6]:
def insert_item(i: int):
    """Блокирующая синхронная функция"""
    conn = sqlite3.connect("demo.db")
    cur = conn.cursor()
    cur.execute("INSERT INTO items (value) VALUES (?)", (f"Item {i}",))
    conn.commit()
    conn.close()
    return i

In [ ]:
# Асинхронное выполнение нескольких блокирующих операций

async def main():
    # Создаем пул потоков
    executor = ThreadPoolExecutor(max_workers=4)
    # Получаем текущий цикл событий
    loop = asyncio.get_running_loop()

    # Запускаем блокирующие операции в пуле потоков:
    # вставляем 10 элементов в базу данных
    tasks = [
        loop.run_in_executor(executor, insert_item, i)
        for i in range(10)
    ]

    # Ожидаем завершения всех задач
    results = await asyncio.gather(*tasks)
    print("Готово:", results)

await main()

Готово: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Пример 2

In [8]:
import numpy as np
import asyncio
from concurrent.futures import ThreadPoolExecutor

In [9]:
def heavy_numpy(n: int):
    """Тяжелое вычисление в NumPy, GIL освобождается"""
    a = np.random.rand(n, n)
    b = np.random.rand(n, n)
    c = a @ b
    return float(c.mean())

In [10]:
async def main():
    # Создаем пул потоков
    executor = ThreadPoolExecutor(max_workers=4)
    # Получаем текущий цикл событий
    loop = asyncio.get_running_loop()

    # Запускаем тяжелые вычисления в пуле потоков:
    # выполняем 4 тяжелые задачи параллельно
    tasks = [
        loop.run_in_executor(executor, heavy_numpy, 1200)
        for _ in range(4)
    ]

    # Ожидаем завершения всех задач
    results = await asyncio.gather(*tasks)
    print("Результаты:", results)

await main()

Результаты: [299.78710612751865, 300.34833689914103, 299.596943185465, 299.61730828726365]


## Пример 3

In [1]:
import cv2
import asyncio
from concurrent.futures import ThreadPoolExecutor

In [2]:
def process_image(path: str):
    """Блокирующая функция: читает и обрабатывает изображение"""
    img = cv2.imread(path)           # блокирующее обращение к диску
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (11, 11), 0)
    return blur.mean()               # возвращаем простую метрику

In [ ]:
async def main():
    # Создаем пул потоков
    loop = asyncio.get_running_loop()
    #
    executor = ThreadPoolExecutor(max_workers=4)

    # Запускаем 4 параллельные обработки изображения
    tasks = [
        loop.run_in_executor(executor, process_image, "test.png")
        for _ in range(4)
    ]

    results = await asyncio.gather(*tasks)
    print("Результаты обработки:", results)

await main()

Результаты обработки: [np.float64(84.04730892181396), np.float64(84.04730892181396), np.float64(84.04730892181396), np.float64(84.04730892181396)]
